In [1]:

from backend.RagCore.Retrieving.retriever import RAGRetriever
retriever = RAGRetriever()
#all_data = retriever.chroma.get(
#    include=["documents", "metadatas", "embeddings"]
#)
#num_docs = len(all_data["documents"])
#print(f"Collection contains {num_docs} documents")
#for doc, meta in zip(all_data["documents"], all_data["metadatas"]):
#    print(f" • {doc[:5]}…  — metadata: {meta}")

INFO: Toutes les variables d'environnement requises sont chargées
INFO: Using ChromaDB path: /Users/mtis/Local/Code/GitRepos/LangAI/data/chromaDB
INFO: Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
/Users/mtis/Local/Code/GitRepos/LangAI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: Load pretrained SentenceTransformer: antoinelouis/french-gte-multilingual-base
INFO: Evaluation mode enabled
INFO: RAGRetriever initialized.


In [3]:
import json
from pathlib import Path
from tqdm import tqdm
import time
import logging

# Setup logger
logger = logging.getLogger("auto_eval")
logger.setLevel(logging.INFO)
if not logger.hasHandlers():
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s"))
    logger.addHandler(h)

def auto_eval(
    retriever : RAGRetriever, 
    dataset_path: str, 
    target: str, 
    output_path: str = "results.json"
):
    """
    Évalue le RAG sur un fichier JSONL filtré par file_name.

    - rag        : instance de RAGRetriever
    - dataset_path    : chemin vers un .jsonl où chaque ligne est un JSON
    - target_file_name: on ne garde que les entrées dont entry["file_name"] == target_file_name
    - top_k      : nombre de docs à récupérer
    - output_path: où écrire les résultats agrégés
    """
    # Lecture du dataset
    with open(dataset_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    # Filtre par file_name on pourra trouver d'autre façons
    filtered = [e for e in data if target in e.get("file_name")]
    logger.info(f"{len(filtered)} questions pour '{target}'")

    results = []
    for entry in tqdm(filtered, desc="Évaluation"):
        question = entry["question"]
        # par défaut on part d'une info vide
        eval_info = {}
        try:
            # Votre méthode answer ne prend plus que question
            answer = retriever.answer(question)
            #logger.info(f"Réponse : {answer}")
            # récupère les métriques stockées dans rag.eval_info
            eval_info = getattr(retriever, "eval_info", {}) or {}
        except Exception as e:
            logger.warning(f"Échec question [{question[:50]}…] — {e}")
            answer = "ERREUR: " + str(e)
            # même en cas d'erreur on garde eval_info vide

        # On stocke tout dans le résultat
        results.append({
            "question": question,
            "expected_source": entry.get("source"),
            "answer": answer,
            "file_name": entry.get("file_name"),
            "top_k": eval_info.get("top_k"),
            "mean_score":    eval_info.get("mean_score"),
            "top1_score":    eval_info.get("top1_score"),
            "temperature":   eval_info.get("temperature"),
            "gen_model":     eval_info.get("gen_model"),
            "provider":      eval_info.get("provider"),
            "context_length": eval_info.get("context_length_chars"),
            "embedding_model": eval_info.get("embedding_model"),
        })

        # petit sleep si besoin de respecter un rate-limit
        time.sleep(15)

    # On écrit le JSON final
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    logger.info(f"Résultats sauvegardés dans '{output_path}'")

In [4]:
auto_eval(retriever,dataset_path="data/questions_strat1.jsonl",target="1881-01-20")

INFO: 18 questions pour '1881-01-20'
Batches: 100%|██████████| 1/1 [00:00<00:00, 13.01it/s]
INFO: → Retrieved 3 docs for: Selon M. Cuneo d'Ornano, combien de fois la Chambre a-t-elle procédé à la constitution de son bureau ordinaire depuis novembre 1877?
INFO: Deduplicating 3 documents...
INFO: 3 unique documents retained.
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]
INFO: HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 11.78it/s]
INFO: → Retrieved 3 docs for: Selon M. Cuneo d'Ornano, pourquoi est-ce qu'il demande l'ajournement des scrutins pour la constitution du bureau?
INFO: Deduplicating 3 documents...
INFO: 3 unique documents retained.
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.38it/s]
INFO: HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 12.55it/s]
INFO: → Retrieved 3 docs for: Quelle est la position de M. Cuneo d'Or

In [6]:
import json
from typing import List

def concat_json_files(output: str, *input_files: List[str]):
    merged = []

    for path in input_files:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError(f"Le fichier '{path}' ne contient pas une liste JSON au niveau supérieur.")
            merged.extend(data)

    with open(output, 'w', encoding='utf-8') as fout:
        json.dump(merged, fout, indent=2, ensure_ascii=False)

    print(f" {len(merged)} éléments sauvegardés dans '{output}' à partir de {len(input_files)} fichiers.")

# Exemple d'utilisation
concat_json_files("dataset.json", "result1.json", "result2.json", "results.json")

 54 éléments sauvegardés dans 'dataset.json' à partir de 3 fichiers.


In [ ]:
import os
print("Clé chargée :", os.getenv("OPENAI_API_KEY") is not None)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
# Configuration de style globale
sns.set(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

# Chargement du fichier JSONL
with open("results.json", "r", encoding="utf-8") as f:
    #data = [json.loads(line) for line in f]
    data = json.load(f)

df = pd.DataFrame(data)

# Nettoyage éventuel
df.rename(columns=lambda x: x.strip(), inplace=True)
if 'context_lenght (chars)' in df.columns:
    df.rename(columns={"context_lenght (chars)": "context_length"}, inplace=True)

# 1. Histogramme des scores moyens
plt.figure()
sns.histplot(df["mean_score"], bins=20, kde=True)
plt.title("Distribution des scores moyens (mean_score)")
plt.xlabel("Score moyen")
plt.ylabel("Fréquence")
plt.show()

# 2. Nuage de points : top1_score vs. context_length
plt.figure()
sns.scatterplot(x="context_length", y="top1_score", hue="provider", data=df)
plt.title("Top1 Score vs Longueur du contexte")
plt.xlabel("Taille du contexte (caractères)")
plt.ylabel("Score du document top-1")
plt.legend()
plt.show()

# 3. Boxplot : score moyen par modèle de génération
plt.figure()
sns.boxplot(x="gen_model", y="mean_score", data=df)
plt.title("Score moyen par modèle de génération")
plt.xlabel("Modèle")
plt.ylabel("Score moyen")
plt.xticks(rotation=45)
plt.show()

# 4. Heatmap : corrélation entre variables numériques
plt.figure()
corr = df[["mean_score", "top1_score", "context_length", "top_k"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Corrélations entre variables")
plt.show()

# 5. Score moyen par date (si file_name contient des dates)
if "file_name" in df.columns:
    df["date"] = pd.to_datetime(df["file_name"], errors="coerce")
    if df["date"].notnull().any():
        df_sorted = df.sort_values("date")
        plt.figure()
        sns.lineplot(x="date", y="mean_score", data=df_sorted)
        plt.title("Évolution du score moyen au fil du temps")
        plt.xlabel("Date")
        plt.ylabel("Score moyen")
        plt.xticks(rotation=45)
        plt.show()

# 6. Nuage de points : top_k vs mean_score
plt.figure()
sns.scatterplot(x="top_k", y="mean_score", hue="provider", data=df)
plt.title("Score moyen en fonction du top_k")
plt.xlabel("top_k")
plt.ylabel("Score moyen")
plt.show()

